# Assignment 1 - LLM Evaluation: Product Description Generation

**Course:** AI Performance Engineering  
**Due Date:** April 5, 2026

This notebook contains the complete solution for Assignment 1, covering:
1. Rubric definition
2. Description generation
3. Manual evaluation
4. Improvement cycle
5. Judge model creation
6. Judge analysis and comparison

---
## Task 1: Define Your Rubric (15 points)

Before generating or evaluating anything, we need a clear, repeatable scoring framework.

### 1.1 Criterion Definitions

For each criterion, we define explicit standards for **good**, **ok**, and **bad** ratings.

#### Fluency
- **Good:** Natural, smooth sentences with varied structure. Easy to read aloud. No awkward phrasing or repetition.
- **Ok:** Mostly natural but with minor awkwardness (e.g., one slightly repetitive phrase or choppy transition).
- **Bad:** Multiple awkward phrases, unnatural word order, or repetitive structure that disrupts readability.

#### Grammar
- **Good:** Zero spelling or punctuation errors. Proper sentence structure throughout.
- **Ok:** One minor error (e.g., missing comma, minor typo) that doesn't affect comprehension.
- **Bad:** Multiple errors or one major error (e.g., subject-verb disagreement, misspelled product name).

#### Tone
- **Good:** Consistently friendly, credible sales voice. Enthusiastic without being pushy. Professional language appropriate for e-commerce.
- **Ok:** Generally appropriate tone but with one instance of overly casual language, excessive hype, or slightly flat delivery.
- **Bad:** Inappropriate tone (too formal/technical, too casual, or overly aggressive sales language). Multiple tone inconsistencies.

#### Length
- **Good:** 50-90 words (inclusive).
- **Ok:** 40-49 words OR 91-110 words.
- **Bad:** Fewer than 40 words OR more than 110 words.

#### Grounding
- **Good:** All information comes directly from provided data (name, attributes, material, warranty). No fabricated features or specifications.
- **Ok:** Minor embellishment that's reasonable inference (e.g., "sleek design" when material is "aluminum") but no false claims.
- **Bad:** Contains fabricated information, incorrect specifications, or claims not supported by the provided data.

#### Latency (avg. time per call)
- **Good:** ≤ 2000ms (2 seconds)
- **Ok:** 2001-5000ms (2-5 seconds)
- **Bad:** > 5000ms (5+ seconds)

#### Cost (avg. price per call)
- **Good:** ≤ $0.01 per description
- **Ok:** $0.011-$0.05 per description
- **Bad:** > $0.05 per description

### 1.2 Pass/Fail Definition

#### Cumulative Pass Bar
A description **passes** if it meets ALL of the following:
- At least **4 "good"** ratings across all criteria
- At most **1 "bad"** rating
- At least **5 "good" or "ok"** ratings combined

#### Go/No-Go Rules (Automatic Failure)
A description **automatically fails** if ANY of these conditions are met:
1. **Grounding is "bad"** - Fabricated information is unacceptable in e-commerce
2. **Grammar is "bad"** - Multiple errors damage credibility
3. **Length is "bad"** - Too far outside target range (< 40 or > 110 words)

#### Pass/Fail Formula
```python
def calculate_pass_fail(ratings: dict) -> str:
    """
    ratings: dict with keys ['fluency', 'grammar', 'tone', 'length', 'grounding', 'latency', 'cost']
    values: 'good', 'ok', or 'bad'
    """
    # Go/no-go rules
    if ratings['grounding'] == 'bad':
        return 'fail'
    if ratings['grammar'] == 'bad':
        return 'fail'
    if ratings['length'] == 'bad':
        return 'fail'
    
    # Cumulative pass bar
    good_count = sum(1 for v in ratings.values() if v == 'good')
    bad_count = sum(1 for v in ratings.values() if v == 'bad')
    good_or_ok_count = sum(1 for v in ratings.values() if v in ['good', 'ok'])
    
    if good_count >= 4 and bad_count <= 1 and good_or_ok_count >= 5:
        return 'pass'
    else:
        return 'fail'
```

---
## Task 2: Generate Descriptions for Every Product (20 points)

Generate product descriptions using a language model from Nebius Token Factory.

In [ ]:
# Import required libraries
import pandas as pd
import httpx
import time
from pathlib import Path
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()

NEBIUS_API_KEY = os.getenv('NEBIUS_API_KEY')
NEBIUS_API_BASE_URL = os.getenv('NEBIUS_API_BASE_URL', 'https://api.studio.nebius.ai/v1')

In [ ]:
# Load the product dataset
df = pd.read_csv('Assignment_01_product_dataset.csv')
print(f"Loaded {len(df)} products")
df.head()

### 2.1 System Prompt

Design a prompt that instructs the model to generate persuasive 50-90 word product descriptions.

In [ ]:
SYSTEM_PROMPT = """
You are an expert e-commerce copywriter. Your task is to write persuasive product descriptions for online shoppers.

Requirements:
- Length: Exactly 50-90 words
- Tone: Friendly, credible, and enthusiastic (but not pushy)
- Content: Use ONLY the provided product information - do not fabricate features
- Style: Natural, easy-to-read sentences with varied structure
- Grammar: Perfect spelling and punctuation

Focus on benefits and appeal to the target customer. Make them want to buy!
""".strip()

def create_user_prompt(product_name: str, attributes: str, material: str, warranty: str) -> str:
    return f"""Product Name: {product_name}
Attributes: {attributes}
Material: {material}
Warranty: {warranty}

Write a persuasive product description (50-90 words)."""

### 2.2 Model Selection

Choose one model from Nebius Token Factory:
- Gemma-2-9b-it
- Meta-Llama-3.1-8B-Instruct

In [ ]:
# TODO: Choose your model
MODEL_NAME = "meta-llama/Meta-Llama-3.1-8B-Instruct"  # or "google/gemma-2-9b-it"

def generate_description(product_name: str, attributes: str, material: str, warranty: str) -> dict:
    """
    Generate a product description and collect metrics.
    
    Returns:
        dict with keys: generated_description, latency_ms, input_tokens, output_tokens
    """
    user_prompt = create_user_prompt(product_name, attributes, material, warranty)
    
    start_time = time.time()
    
    # TODO: Implement API call to Nebius Token Factory
    # Use httpx to make the request
    # Collect: description, input_tokens, output_tokens
    
    response = httpx.post(
        f"{NEBIUS_API_BASE_URL}/chat/completions",
        headers={
            "Authorization": f"Bearer {NEBIUS_API_KEY}",
            "Content-Type": "application/json"
        },
        json={
            "model": MODEL_NAME,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt}
            ],
            "temperature": 0.7,
            "max_tokens": 150
        },
        timeout=30.0
    )
    
    end_time = time.time()
    latency_ms = int((end_time - start_time) * 1000)
    
    response_data = response.json()
    
    return {
        'generated_description': response_data['choices'][0]['message']['content'].strip(),
        'latency_ms': latency_ms,
        'input_tokens': response_data['usage']['prompt_tokens'],
        'output_tokens': response_data['usage']['completion_tokens']
    }

### 2.3 Generate Descriptions for All Products

In [ ]:
# TODO: Run generation for all products
results = []

for idx, row in df.iterrows():
    print(f"Processing {idx+1}/{len(df)}: {row['product_name']}")
    
    result = generate_description(
        product_name=row['product_name'],
        attributes=row['Product_attribute_list'],
        material=row['material'],
        warranty=row['warranty']
    )
    
    # Combine original data with generated results
    results.append({
        **row.to_dict(),
        **result
    })
    
    # Small delay to avoid rate limiting
    time.sleep(0.5)

print("\nGeneration complete!")

### 2.4 Create DataFrame and Save to Excel

In [ ]:
# Create results DataFrame
results_df = pd.DataFrame(results)

# Add blank columns for evaluation criteria
results_df['fluency'] = ''
results_df['grammar'] = ''
results_df['tone'] = ''
results_df['length'] = ''
results_df['grounding'] = ''
results_df['latency'] = ''
results_df['cost'] = ''
results_df['final_score'] = ''

# Save to Excel
results_df.to_excel('assignment_01.xlsx', index=False)
print("Saved results to assignment_01.xlsx")

# Display summary
print(f"\nGenerated {len(results_df)} descriptions")
print(f"Average latency: {results_df['latency_ms'].mean():.0f}ms")
print(f"Average input tokens: {results_df['input_tokens'].mean():.0f}")
print(f"Average output tokens: {results_df['output_tokens'].mean():.0f}")

results_df.head()

---
## Task 3: Manual (Human) Evaluation (10 points)

Manually evaluate 10-15 products using the rubric defined in Task 1.

### 3.1 Add Cost Column

Calculate the cost per description based on token usage and model pricing.

In [ ]:
# TODO: Add pricing information for your chosen model
# Nebius Token Factory pricing (example - verify actual pricing)
PRICE_PER_1K_INPUT_TOKENS = 0.0001  # USD per 1K input tokens
PRICE_PER_1K_OUTPUT_TOKENS = 0.0002  # USD per 1K output tokens

# Load the Excel file
results_df = pd.read_excel('assignment_01.xlsx')

# Calculate cost
results_df['cost_usd'] = (
    (results_df['input_tokens'] / 1000 * PRICE_PER_1K_INPUT_TOKENS) +
    (results_df['output_tokens'] / 1000 * PRICE_PER_1K_OUTPUT_TOKENS)
)

print(f"Average cost per description: ${results_df['cost_usd'].mean():.4f}")
print(f"Total cost for {len(results_df)} descriptions: ${results_df['cost_usd'].sum():.4f}")

### 3.2 Manual Evaluation Instructions

**TODO: Manually evaluate 10-15 products**

1. Open `assignment_01.xlsx` in Excel
2. Select 10-15 diverse products (mix of different categories)
3. For each selected product, rate each criterion (fluency, grammar, tone, length, grounding, latency, cost) as:
   - `good`
   - `ok`
   - `bad`
4. Use the rubric definitions from Task 1
5. Calculate `final_score` (pass/fail) using the formula from Task 1
6. Save the Excel file

After completing manual evaluation, run the cell below to load and analyze your scores.

In [ ]:
# Load manually evaluated data
evaluated_df = pd.read_excel('assignment_01.xlsx')

# Filter rows with manual evaluation (non-empty fluency column)
manual_eval = evaluated_df[evaluated_df['fluency'] != ''].copy()

print(f"Manually evaluated {len(manual_eval)} products\n")

# Analyze baseline performance
criteria = ['fluency', 'grammar', 'tone', 'length', 'grounding', 'latency', 'cost']

print("Baseline Analysis - Criterion Performance:")
print("=" * 50)

for criterion in criteria:
    if criterion in manual_eval.columns:
        value_counts = manual_eval[criterion].value_counts()
        print(f"\n{criterion.upper()}:")
        for rating in ['good', 'ok', 'bad']:
            count = value_counts.get(rating, 0)
            pct = (count / len(manual_eval) * 100) if len(manual_eval) > 0 else 0
            print(f"  {rating}: {count} ({pct:.1f}%)")

# Pass/fail summary
if 'final_score' in manual_eval.columns:
    final_counts = manual_eval['final_score'].value_counts()
    print(f"\n{'='*50}")
    print("FINAL SCORES:")
    print(f"  Pass: {final_counts.get('pass', 0)}")
    print(f"  Fail: {final_counts.get('fail', 0)}")
    pass_rate = (final_counts.get('pass', 0) / len(manual_eval) * 100) if len(manual_eval) > 0 else 0
    print(f"  Pass rate: {pass_rate:.1f}%")

### 3.3 Baseline Analysis

**TODO: Document your findings**

Based on the manual evaluation:

1. **Best performing criteria:**
   - [Write your analysis here]

2. **Worst performing criteria:**
   - [Write your analysis here]

3. **Common failure patterns:**
   - [Write your analysis here]

4. **Strategy for improvement (Task 4):**
   - [Write your improvement strategy here]

---
## Task 4: Improvement Cycle (15 points)

Iterate to achieve better results based on Task 3 baseline analysis.

### Experiment Template

For each experiment, document:
1. **What you changed**
2. **Why you expected it to help**
3. **New evaluation scores**

Keep code for successful experiments. Document failed experiments but code is optional.

### Experiment 1: [Name your experiment]

**What changed:**
- [Describe the change]

**Why expected to help:**
- [Explain your hypothesis]

**Results:**
- [Document the outcome]

In [ ]:
# TODO: Experiment 1 code
# Example: Modified prompt, different temperature, etc.

### Experiment 2: [Name your experiment]

**What changed:**
- [Describe the change]

**Why expected to help:**
- [Explain your hypothesis]

**Results:**
- [Document the outcome]

In [ ]:
# TODO: Experiment 2 code

### Experiment 3: [Name your experiment]

**What changed:**
- [Describe the change]

**Why expected to help:**
- [Explain your hypothesis]

**Results:**
- [Document the outcome]

In [ ]:
# TODO: Experiment 3 code

---
## Task 5: Create a Judge Model (20 points)

Build an automated LLM judge that grades descriptions using the Task 1 rubric.

### 5.1 Judge Model Selection

Start with the model you **did not** use in Task 2. If it struggles, switch to a larger model.

In [ ]:
# TODO: Choose judge model (the one NOT used in Task 2)
JUDGE_MODEL_NAME = "google/gemma-2-9b-it"  # or another model if needed

print(f"Judge model: {JUDGE_MODEL_NAME}")
print(f"Generator model was: {MODEL_NAME}")

### 5.2 Pydantic Schema for Structured Output

Define the output schema. Note: **explanation comes before verdict** (important for chain-of-thought reasoning).

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class CriterionEvaluation(BaseModel):
    explanation: str = Field(description="Reasoning for the verdict")
    verdict: Literal['good', 'ok', 'bad'] = Field(description="Rating: good, ok, or bad")

class DescriptionEvaluation(BaseModel):
    fluency: CriterionEvaluation
    grammar: CriterionEvaluation
    tone: CriterionEvaluation
    length: CriterionEvaluation
    grounding: CriterionEvaluation

# Display schema
print("Judge output schema:")
print(DescriptionEvaluation.model_json_schema())

**Why explanation before verdict?**

[TODO: Explain why this ordering matters for LLM reasoning]

### 5.3 Judge Prompt

Write a prompt that embeds the Task 1 rubric and provides necessary context for evaluation.

In [ ]:
JUDGE_SYSTEM_PROMPT = """
You are an expert evaluator of product descriptions. Your task is to rate product descriptions according to specific criteria.

For each criterion, provide:
1. An explanation of your reasoning
2. A verdict: 'good', 'ok', or 'bad'

EVALUATION CRITERIA:

FLUENCY:
- good: Natural, smooth sentences with varied structure. Easy to read aloud. No awkward phrasing or repetition.
- ok: Mostly natural but with minor awkwardness (e.g., one slightly repetitive phrase or choppy transition).
- bad: Multiple awkward phrases, unnatural word order, or repetitive structure that disrupts readability.

GRAMMAR:
- good: Zero spelling or punctuation errors. Proper sentence structure throughout.
- ok: One minor error (e.g., missing comma, minor typo) that doesn't affect comprehension.
- bad: Multiple errors or one major error (e.g., subject-verb disagreement, misspelled product name).

TONE:
- good: Consistently friendly, credible sales voice. Enthusiastic without being pushy. Professional language appropriate for e-commerce.
- ok: Generally appropriate tone but with one instance of overly casual language, excessive hype, or slightly flat delivery.
- bad: Inappropriate tone (too formal/technical, too casual, or overly aggressive sales language). Multiple tone inconsistencies.

LENGTH:
- good: 50-90 words (inclusive)
- ok: 40-49 words OR 91-110 words
- bad: Fewer than 40 words OR more than 110 words

GROUNDING:
- good: All information comes directly from provided product data. No fabricated features or specifications.
- ok: Minor embellishment that's reasonable inference but no false claims.
- bad: Contains fabricated information, incorrect specifications, or claims not supported by the provided data.

Be objective and consistent in your evaluations.
""".strip()

def create_judge_prompt(description: str, product_name: str, attributes: str, material: str, warranty: str) -> str:
    return f"""PRODUCT INFORMATION:
Name: {product_name}
Attributes: {attributes}
Material: {material}
Warranty: {warranty}

GENERATED DESCRIPTION:
{description}

Evaluate this description according to the criteria (fluency, grammar, tone, length, grounding)."""

print("Judge prompt created")

### 5.4 Judge Implementation

In [ ]:
def judge_description(description: str, product_name: str, attributes: str, material: str, warranty: str) -> DescriptionEvaluation:
    """
    Use the judge model to evaluate a product description.
    
    Returns:
        DescriptionEvaluation object with ratings for each criterion
    """
    user_prompt = create_judge_prompt(description, product_name, attributes, material, warranty)
    
    # TODO: Implement API call with structured output
    response = httpx.post(
        f"{NEBIUS_API_BASE_URL}/chat/completions",
        headers={
            "Authorization": f"Bearer {NEBIUS_API_KEY}",
            "Content-Type": "application/json"
        },
        json={
            "model": JUDGE_MODEL_NAME,
            "messages": [
                {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt}
            ],
            "temperature": 0.3,  # Lower temperature for more consistent evaluation
            "response_format": {
                "type": "json_schema",
                "json_schema": {
                    "name": "description_evaluation",
                    "schema": DescriptionEvaluation.model_json_schema()
                }
            }
        },
        timeout=30.0
    )
    
    response_data = response.json()
    content = response_data['choices'][0]['message']['content']
    
    # Parse into Pydantic model
    return DescriptionEvaluation.model_validate_json(content)

print("Judge function ready")

---
## Task 6: Run and Analyze the Judge (20 points)

Run the judge model and compare its evaluations to human ratings.

### 6.1 Sanity Check (5 products)

In [ ]:
# TODO: Run judge on 5 products for sanity check
results_df = pd.read_excel('assignment_01.xlsx')

# Select 5 products (can be random or specific)
sanity_check_indices = [0, 10, 20, 30, 40]  # Adjust as needed

print("Sanity Check - Judge Evaluations:")
print("=" * 80)

for idx in sanity_check_indices:
    row = results_df.iloc[idx]
    
    print(f"\nProduct: {row['product_name']}")
    print(f"Description: {row['generated_description'][:100]}...")
    
    evaluation = judge_description(
        description=row['generated_description'],
        product_name=row['product_name'],
        attributes=row['Product_attribute_list'],
        material=row['material'],
        warranty=row['warranty']
    )
    
    print("\nJudge Ratings:")
    for criterion in ['fluency', 'grammar', 'tone', 'length', 'grounding']:
        eval_obj = getattr(evaluation, criterion)
        print(f"  {criterion}: {eval_obj.verdict}")
        print(f"    → {eval_obj.explanation}")
    
    time.sleep(1)  # Rate limiting

print("\n" + "=" * 80)
print("Review the explanations and verdicts above.")
print("Does the judge apply your rubric correctly? Adjust prompt if needed.")

**Sanity Check Analysis:**

[TODO: Review the judge outputs above. Do they make sense? Does the judge apply your rubric correctly? Document any issues and prompt adjustments needed.]

### 6.2 Full Run - Judge All Products

In [ ]:
# TODO: Run judge on all products
results_df = pd.read_excel('assignment_01.xlsx')

# Add judge columns
judge_columns = [
    'judge_fluency', 'judge_fluency_explanation',
    'judge_grammar', 'judge_grammar_explanation',
    'judge_tone', 'judge_tone_explanation',
    'judge_length', 'judge_length_explanation',
    'judge_grounding', 'judge_grounding_explanation',
    'judge_final_score'
]

for col in judge_columns:
    if col not in results_df.columns:
        results_df[col] = ''

print(f"Running judge on {len(results_df)} products...")

for idx, row in results_df.iterrows():
    print(f"Judging {idx+1}/{len(results_df)}: {row['product_name']}")
    
    evaluation = judge_description(
        description=row['generated_description'],
        product_name=row['product_name'],
        attributes=row['Product_attribute_list'],
        material=row['material'],
        warranty=row['warranty']
    )
    
    # Store judge ratings
    for criterion in ['fluency', 'grammar', 'tone', 'length', 'grounding']:
        eval_obj = getattr(evaluation, criterion)
        results_df.at[idx, f'judge_{criterion}'] = eval_obj.verdict
        results_df.at[idx, f'judge_{criterion}_explanation'] = eval_obj.explanation
    
    # Calculate judge final score using Task 1 formula
    judge_ratings = {
        'fluency': results_df.at[idx, 'judge_fluency'],
        'grammar': results_df.at[idx, 'judge_grammar'],
        'tone': results_df.at[idx, 'judge_tone'],
        'length': results_df.at[idx, 'judge_length'],
        'grounding': results_df.at[idx, 'judge_grounding'],
        'latency': results_df.at[idx, 'latency'],  # From Task 2
        'cost': results_df.at[idx, 'cost']  # From Task 3
    }
    
    # Apply pass/fail formula from Task 1
    # TODO: Implement calculate_pass_fail function or inline logic
    results_df.at[idx, 'judge_final_score'] = 'pass'  # Placeholder
    
    time.sleep(1)  # Rate limiting

# Save updated results
results_df.to_excel('assignment_01.xlsx', index=False)
print("\nJudge evaluation complete! Results saved to assignment_01.xlsx")

### 6.3 Compare Judge vs Human Evaluation

In [ ]:
# Load results with both human and judge evaluations
results_df = pd.read_excel('assignment_01.xlsx')

# Filter rows with human evaluation
compared = results_df[results_df['fluency'] != ''].copy()

print(f"Comparing judge vs human on {len(compared)} products\n")
print("Agreement Rates by Criterion:")
print("=" * 50)

criteria = ['fluency', 'grammar', 'tone', 'length', 'grounding']

for criterion in criteria:
    human_col = criterion
    judge_col = f'judge_{criterion}'
    
    if human_col in compared.columns and judge_col in compared.columns:
        agreements = (compared[human_col] == compared[judge_col]).sum()
        total = len(compared)
        agreement_rate = (agreements / total * 100) if total > 0 else 0
        
        print(f"\n{criterion.upper()}:")
        print(f"  Agreement: {agreements}/{total} ({agreement_rate:.1f}%)")
        
        # Show disagreements
        disagreements = compared[compared[human_col] != compared[judge_col]]
        if len(disagreements) > 0:
            print(f"  Disagreements:")
            for _, row in disagreements.iterrows():
                print(f"    - {row['product_name'][:40]}: Human={row[human_col]}, Judge={row[judge_col]}")

# Overall agreement
print(f"\n{'='*50}")
print("Overall Analysis:")
# TODO: Calculate overall agreement and analyze patterns

**Analysis of Judge vs Human Agreement:**

[TODO: Document your findings]

1. **Where do they agree most?**
   - 

2. **Where do they diverge?**
   - 

3. **Why might these differences occur?**
   - 

### 6.4 Criterion-by-Criterion Judging

Run the judge separately for each criterion (one API call per criterion per product).

In [ ]:
# TODO: Implement single-criterion judge
def judge_single_criterion(description: str, product_name: str, attributes: str, 
                          material: str, warranty: str, criterion: str) -> CriterionEvaluation:
    """
    Judge a single criterion in isolation.
    
    Args:
        criterion: One of 'fluency', 'grammar', 'tone', 'length', 'grounding'
    """
    # Create criterion-specific prompt
    criterion_prompt = f"""PRODUCT INFORMATION:
Name: {product_name}
Attributes: {attributes}
Material: {material}
Warranty: {warranty}

GENERATED DESCRIPTION:
{description}

Evaluate ONLY the {criterion.upper()} of this description according to the rubric."""
    
    # TODO: Implement API call for single criterion
    # Similar to judge_description but returns only CriterionEvaluation
    pass

# Run criterion-by-criterion evaluation on a subset
# TODO: Implement and compare results

**Criterion-by-Criterion Analysis:**

[TODO: Answer these questions]

1. **Did isolating criteria change the results?**
   - 

2. **Why might this approach lead to different outcomes?**
   - 

3. **Did agreement with human scores improve?**
   - 

### 6.5 Final Analysis and Reflection

#### Question 1: Trade-offs between human evaluation and LLM-as-a-judge

Consider: cost, scale, consistency, accuracy

[TODO: Write your analysis here]

**Human Evaluation:**
- Pros:
  - 
- Cons:
  - 

**LLM-as-a-Judge:**
- Pros:
  - 
- Cons:
  - 

#### Question 2: Recommendation for production system

For a production system generating thousands of descriptions daily:

[TODO: Write your recommendation here]

**Recommended approach:**
- 

**Justification:**
- 

**Implementation considerations:**
- 

---
## Summary and Submission

### Deliverables Checklist

- [ ] Task 1: Rubric definitions and pass/fail formula
- [ ] Task 2: Code for description generation
- [ ] Task 3: `assignment_01.xlsx` with manual evaluations (10-15 products)
- [ ] Task 4: Experiment documentation + code for successful experiments
- [ ] Task 5: Judge model implementation with Pydantic schema
- [ ] Task 6: Judge analysis, comparisons, and reflection

### Files to Submit

1. `assignment_01_solution.ipynb` (this notebook)
2. `assignment_01.xlsx` (with all evaluations)

**Due Date:** April 5, 2026